# 13 · Frozen-checkpoint benchmark and generalization
**Status: UNEXECUTED FIRST DRAFT.** No benchmark score is claimed.

**Purpose:** measure frozen agents on explicitly held-out public developer game families using only the official public observation/action interface. **Inputs:** a trained checkpoint, configured disjoint split manifest, ARC toolkit environment access and fixed action/compute budgets. **Outputs:** per-game traces, completion/action/time measurements, a raw toolkit scorecard when returned, and a reproducibility report.

**Expected runtime:** a configurable wall limit (default 15 minutes per invocation) plus finishing/closing work. Throughput is unmeasured. **Checkpoint directory:** `checkpoints/<benchmark.run_id>/`; progress includes frozen weight/source/config hashes, completed games and interrupted episode identity. The original model checkpoint supplies immutable weights; this read-only stage has no optimizer.

**Resume:** completed games are never silently repeated. A crash in an active game cannot restore hidden environment state through an unsupported API. The default refuses to resume an interrupted episode; setting `benchmark.interrupted_episode='discard_and_restart'` records that choice and restarts only that developer episode from its beginning. Its abandoned actions remain recorded. This is not exact online/competition continuation.

**Run All:** evaluates only when an explicit checkpoint and held-out game list exist. This notebook does not enter competition mode, accept rules or submit anything. Public held-out games provide a development generalization test, not evidence about secret competition games. Raw official scorecards are kept separately from local proxy metrics; no human-relative official score is reconstructed from guessed baselines.

## Load definitions

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([12], globals())


## Freeze the model and normalize trace values

In [ ]:
def json_safe(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().tolist()
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if hasattr(value, 'tolist'):
        return value.tolist()
    if hasattr(value, 'model_dump'):
        return json_safe(value.model_dump(mode='json'))
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if hasattr(value, 'value'):
        return json_safe(value.value)
    return str(value)


def load_ahr_for_evaluation(checkpoint_path, cfg=None, device='cpu', root=ROOT):
    payload, saved_cfg, identity = read_ahr_checkpoint(checkpoint_path, root)
    if identity['step'] < 1:
        raise ValueError('Benchmark requires a checkpoint after at least one training update.')
    model_cfg = copy.deepcopy(saved_cfg if cfg is None else cfg)
    if model_cfg['model'] != saved_cfg['model']:
        raise ValueError('Evaluation model dimensions differ from the trained checkpoint.')
    model = AHRModel(model_cfg).to(device)
    model.load_state_dict(payload['model'], strict=True)
    model.eval()
    model.checkpoint_identity = identity
    model.trained_config = copy.deepcopy(payload.get('extra', {}).get('trained_config', saved_cfg))
    families = {game.split('-', 1)[0] for game in model.trained_config.get('data', {}).get('train_games', [])}
    families.update(payload.get('extra', {}).get('training_game_families', []))
    model.training_game_families = sorted(families)
    return model


def benchmark_state_terminal(obs):
    state = str(obs.get('state', '')).upper().split('.')[-1]
    return state in {'WIN', 'GAME_OVER', 'COMPLETE', 'COMPLETED'}


def _evaluation_progress_records(directory):
    directory = Path(directory)
    candidates = {}
    manifest_path = directory / 'manifest.json'
    if manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text())
            for key in ('latest', 'previous', 'best'):
                name, digest = manifest.get(key), manifest.get(key + '_sha256')
                if name and digest:
                    candidates[name] = {'file': name, 'sha256': digest, 'saved_ns': 0}
        except (OSError, ValueError, TypeError):
            # Individually committed immutable progress files can repair a corrupt manifest.
            pass
    for marker in directory.glob('progress-*.commit.json'):
        try:
            item = json.loads(marker.read_text())
            if isinstance(item, dict) and 'file' in item:
                candidates[item['file']] = item
        except (OSError, ValueError, TypeError):
            continue
    valid = []
    for item in candidates.values():
        name = item.get('file')
        if not isinstance(name, str) or Path(name).name != name:
            continue
        path = directory / name
        try:
            if not path.is_file() or sha256_file(path) != item.get('sha256'):
                continue
            state = json.loads(path.read_text())
            if not isinstance(state, dict) or not {'protocol_hash', 'model_identity', 'completed', 'active_game'}.issubset(state):
                continue
        except (OSError, ValueError, TypeError):
            continue
        valid.append((item, state))
    valid.sort(key=lambda pair: int(pair[0].get('saved_ns', 0)), reverse=True)
    return valid


def write_evaluation_progress(directory, progress):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    previous = _evaluation_progress_records(directory)
    if previous and previous[0][1]['protocol_hash'] != progress['protocol_hash']:
        raise ValueError('Evaluation journal protocol changed; use a new run ID.')
    stamp = time.time_ns()
    name = f'progress-{stamp}.json'
    path = directory / name
    atomic_json(path, progress)
    item = {'file': name, 'sha256': sha256_file(path), 'saved_ns': stamp,
            'protocol_hash': progress['protocol_hash']}
    # The commit marker allows recovery after the data file is durable but before the manifest is replaced.
    atomic_json(directory / f'progress-{stamp}.commit.json', item)
    retained = [(item, progress)] + previous[:1]
    best = (item, progress) if progress.get('status') == 'complete' else next((pair for pair in previous if pair[1].get('status') == 'complete'), None)
    if best and best[0]['file'] not in {pair[0]['file'] for pair in retained}:
        retained.append(best)
    prior = previous[0][0] if previous else None
    manifest = {'latest': name, 'latest_sha256': item['sha256'],
                'previous': prior['file'] if prior else None, 'previous_sha256': prior['sha256'] if prior else None,
                'best': best[0]['file'] if best else None, 'best_sha256': best[0]['sha256'] if best else None,
                'model_checkpoint': progress['model_identity'], 'resume_notebook': '13_benchmark_and_generalization.ipynb',
                'resume_policy': 'completed episodes exact; interrupted developer episode requires explicit discard_and_restart'}
    atomic_json(directory / 'manifest.json', manifest)
    keep = {pair[0]['file'] for pair in retained}
    for record, _ in previous:
        if record['file'] not in keep:
            (directory / record['file']).unlink(missing_ok=True)
            (directory / (Path(record['file']).stem + '.commit.json')).unlink(missing_ok=True)


def read_evaluation_progress(directory):
    directory = Path(directory)
    valid = _evaluation_progress_records(directory)
    if valid:
        return valid[0][1]
    markers = list(directory.glob('progress-*.json'))
    legacy = any((directory / name).exists() for name in ('manifest.json', 'latest.json', 'previous.json', 'best.json'))
    if markers or legacy:
        raise RuntimeError('No verified evaluation progress remains; refusing silent restart. Inspect the journal and preserve its evidence.')
    return None


def direct_policy_action(model, obs):
    device = next(model.parameters()).device
    grid = torch.as_tensor(obs['grid'], dtype=torch.long, device=device)[None]
    with torch.no_grad():
        output = model(grid)
        mask = torch.zeros((1, 8), dtype=torch.bool, device=device)
        for action in obs['available_actions']:
            mask[0, int(action)] = True
        action = int(legal_policy(output['action_logits'], mask).argmax(-1).item())
        xy = output['click_logits'][0].argmax(-1).tolist()
    spec = ActionSpec(action, int(xy[0]) if action == 6 else None, int(xy[1]) if action == 6 else None)
    return spec, {'controller': 'direct learned policy; no search or episodic workspace', 'model_calls': 1}


## Evaluate through the public interface
The direct-policy option is a frozen-weight inference control. It is not a separately trained transformer baseline. All methods receive the same observations and external action cap; actual internal model calls and latency must also be compared.

In [ ]:
def evaluate_checkpoint(checkpoint_path, cfg, split='heldout', root=ROOT):
    cfg = copy.deepcopy(cfg)
    options = cfg.get('benchmark', {})
    mode = str(cfg['environment'].get('operation_mode', 'OFFLINE')).upper()
    if mode != 'OFFLINE':
        raise ValueError('This draft benchmark uses OFFLINE public developer games only. Official competition needs its own reviewed run protocol.')
    if split not in {'dev', 'heldout'}:
        raise ValueError('Benchmark split must be dev or heldout.')
    registry = SplitRegistry.from_config(cfg)
    split_path = Path(cfg['data']['split_manifest'])
    if not split_path.is_absolute():
        split_path = Path(root) / split_path
    if not split_path.is_file() or SplitRegistry.from_manifest(split_path).to_dict() != registry.to_dict():
        raise ValueError('Configured game split must match notebook 03 split manifest.')
    games = list(cfg['data'].get(f'{split}_games', []))
    if not games:
        raise ValueError(f'Configure an explicit {split} game-family list before benchmarking.')
    for game in games:
        registry.assert_game(game, split)
    require_recovery_gate(root)
    require_full_model_gate(cfg, root)
    device = options.get('device', cfg['train']['device'])
    model = load_ahr_for_evaluation(checkpoint_path, cfg, device, root)
    if not model.training_game_families:
        raise ValueError('Checkpoint lacks original training-family provenance; held-out status cannot be established.')
    overlap = sorted({game.split('-', 1)[0] for game in games} & set(model.training_game_families))
    if overlap:
        raise ValueError(f'Evaluation games overlap the checkpoint training lineage: {overlap}. Editing the current split cannot erase training exposure.')
    source = source_manifest(root)
    protocol_cfg = copy.deepcopy(cfg)
    protocol_cfg.get('benchmark', {}).pop('interrupted_episode', None)
    protocol = {'configuration': protocol_cfg, 'split': split, 'games': games, 'model': model.checkpoint_identity,
                'source_hashes': source, 'trained_config': model.trained_config, 'training_game_families': model.training_game_families, 'split_sha256': sha256_file(split_path),
                'learning_during_evaluation': False, 'memory_reset_between_games': True}
    protocol_hash = canonical_hash(protocol)
    run_id = options.get('run_id', f'xl-benchmark-{split}-v1')
    directory = Path(root) / 'checkpoints' / run_id
    max_actions = int(cfg['environment']['max_actions'])
    max_wall = float(options.get('max_wall_seconds', 900))
    if max_actions < 1 or max_wall <= 0:
        raise ValueError('Benchmark action and time limits must be positive.')
    controller = options.get('controller', 'ahr')
    if controller not in {'ahr', 'direct_policy'}:
        raise ValueError('controller must be ahr or direct_policy.')
    with RunLock(root, run_id):
        progress = read_evaluation_progress(directory)
        if progress is not None:
            if progress['protocol_hash'] != protocol_hash:
                raise ValueError('Existing benchmark identity differs. Choose a new run ID.')
            if progress.get('active_game') is not None:
                if options.get('interrupted_episode', 'refuse') != 'discard_and_restart':
                    raise RuntimeError('Interrupted game cannot be exactly restored. Review its trace, then explicitly choose discard_and_restart for a developer rerun.')
                progress.setdefault('discarded_episodes', []).append(progress['active_game'])
                progress['active_game'] = None
            if progress.get('status') == 'complete':
                print('This frozen benchmark is already complete; returning its saved result.')
                return progress
        else:
            progress = {'run_id': run_id, 'protocol_hash': protocol_hash, 'protocol': protocol,
                        'model_identity': model.checkpoint_identity, 'completed': [], 'active_game': None,
                        'discarded_episodes': [], 'status': 'in_progress', 'raw_toolkit_scorecards': []}
        progress['status'] = 'in_progress'
        completed_games = {r['game_id'] for r in progress['completed']}
        started = time.monotonic()
        session = None
        try:
            for game_index, game in enumerate(games):
                if game in completed_games:
                    continue
                if time.monotonic() - started >= max_wall:
                    break
                game_seed = int(cfg['environment'].get('seed', 42)) + game_index
                seed_everything(game_seed)
                if session is not None:
                    scorecard = session.close()
                    if scorecard is not None:
                        progress['raw_toolkit_scorecards'].append(json_safe(scorecard))
                game_cfg = copy.deepcopy(cfg)
                game_cfg['environment']['seed'] = game_seed
                session = ARCSession(game_cfg, operation_mode='OFFLINE')
                episode_state = None
                if hasattr(model, 'reset_episode'):
                    model.reset_episode()
                obs = session.reset(game)
                episode_started = time.monotonic()
                trace_path = Path(root) / 'reports' / 'traces' / f'{run_id}-{game}-{time.time_ns()}.jsonl'
                actions, internal_calls = 0, 0
                initial_levels = int(obs.get('levels_completed', 0))
                inference_seconds = 0.0
                progress['active_game'] = {'game_id': game, 'actions': 0, 'trace': str(trace_path), 'seed': game_seed}
                write_evaluation_progress(directory, progress)
                while actions < max_actions and not benchmark_state_terminal(obs):
                    if time.monotonic() - started >= max_wall:
                        progress['status'] = 'wall_limit_with_interrupted_episode'
                        break
                    decision_started = time.perf_counter()
                    if controller == 'ahr':
                        with torch.no_grad():
                            action, episode_state, trace = model.act(obs, obs['available_actions'], episode_state=episode_state, budget=cfg['reasoning']['max_model_calls'])
                    else:
                        action, trace = direct_policy_action(model, obs)
                    if next(model.parameters()).device.type == 'cuda':
                        torch.cuda.synchronize(next(model.parameters()).device)
                    decision_seconds = time.perf_counter() - decision_started
                    inference_seconds += decision_seconds
                    calls = trace.get('budget', {}).get('used', trace.get('model_calls')) if isinstance(trace, dict) else None
                    if calls is None:
                        raise ValueError('Planner trace lacks measured neural-call count.')
                    internal_calls += int(calls)
                    next_obs = session.step(action)
                    actions += 1
                    append_metrics(trace_path, {'game_id': game, 'step': actions, 'observed_grid': json_safe(obs['grid']),
                                               'action': action.to_dict(), 'next_observed_grid': json_safe(next_obs['grid']),
                                               'levels_completed': next_obs.get('levels_completed'), 'state': json_safe(next_obs.get('state')),
                                               'decision_seconds': decision_seconds,
                                               'planning_trace': json_safe(trace), 'imagined_is_evidence': False})
                    obs = next_obs
                    progress['active_game'].update({'actions': actions, 'episode_state': json_safe(episode_state),
                                                     'last_observation': json_safe(obs), 'internal_calls': internal_calls})
                    # Bounded public-state journal; hidden toolkit state is explicitly not restorable.
                    write_evaluation_progress(directory, progress)
                if progress.get('status') == 'wall_limit_with_interrupted_episode':
                    break
                state = str(obs.get('state', '')).upper().split('.')[-1]
                progress['completed'].append({'game_id': game, 'seed': game_seed, 'split': split, 'won': state == 'WIN', 'state': state,
                    'levels_completed': int(obs.get('levels_completed', 0)) - initial_levels,
                    'actions': actions, 'internal_model_calls': internal_calls, 'inference_seconds': inference_seconds, 'elapsed_seconds': time.monotonic() - episode_started,
                    'trace': str(trace_path), 'stopped_at_action_limit': actions >= max_actions and not benchmark_state_terminal(obs)})
                progress['active_game'] = None
                write_evaluation_progress(directory, progress)
                print(progress['completed'][-1])
        finally:
            if session is not None:
                scorecard = session.close()
                if scorecard is not None:
                    progress['raw_toolkit_scorecards'].append(json_safe(scorecard))
            write_evaluation_progress(directory, progress)
        if source_manifest(root) != source or sha256_file(model.checkpoint_identity['path']) != model.checkpoint_identity['sha256']:
            raise RuntimeError('Source or model checkpoint changed during evaluation; result cannot be treated as frozen.')
        finished = len(progress['completed']) == len(games)
        progress['status'] = 'complete' if finished else progress.get('status', 'in_progress')
        records = progress['completed']
        progress['local_proxy_metrics'] = {'games_completed': len(records), 'games_requested': len(games),
            'win_fraction': sum(r['won'] for r in records) / len(records) if records else None,
            'levels_completed': sum(r['levels_completed'] for r in records), 'total_actions': sum(r['actions'] for r in records),
            'total_inference_seconds': sum(r['inference_seconds'] for r in records)}
        progress['official_arc_prize_score'] = None
        progress['scope'] = 'public developer held-out game families; local proxies are not official RHAE'
        write_evaluation_progress(directory, progress)
        atomic_json(Path(root) / 'reports' / f'{run_id}.json', progress)
        return progress


## Select a trained checkpoint and run the frozen protocol

In [ ]:
cfg = load_config()
cfg.setdefault('benchmark', {})
cfg['benchmark'].setdefault('checkpoint', None)
cfg['benchmark'].setdefault('run_id', 'xl-benchmark-heldout-v1')
cfg['benchmark'].setdefault('controller', 'ahr')
cfg['benchmark'].setdefault('max_wall_seconds', 900)
checkpoint = cfg['benchmark']['checkpoint']
if not checkpoint or not cfg['data'].get('heldout_games'):
    print('BENCHMARK NOT STARTED: set benchmark.checkpoint and explicit heldout_games in the shared configuration.')
else:
    benchmark_report = evaluate_checkpoint(checkpoint, cfg, split='heldout')
    print(benchmark_report['status'], benchmark_report['local_proxy_metrics'])


## Interpretation after execution
Compare paired variants on the same game families and seeds. Report wins, levels, external actions, actual inference time and internal calls separately. A public-development improvement cannot establish hidden-test generalization. Keep failed and truncated games in the denominator when comparing completed campaigns; do not cherry-pick successful traces. The report retains whether each episode hit the external action limit.